# 🕵️ UPI Fraud Rings: starter notebook
**ML Gradient Wars 2026**

This notebook loads the data, looks at it, builds **single-row** features, trains a gradient-boosting model and writes a valid `submission.csv`.

It is a **baseline**. It looks at each payment on its own, so it can't see mule accounts, shared phones, money chains or repeated refunds. It scores low. Beating it is your job.

It runs in a few minutes on a laptop with 8 GB RAM. Put it in the same folder as the data files (or change `DATA` below).

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

DATA = '../data/raw'
train = pd.read_csv(f'{DATA}/train.csv')
test = pd.read_csv(f'{DATA}/test.csv')
accounts = pd.read_csv(f'{DATA}/accounts.csv')
merchants = pd.read_csv(f'{DATA}/merchants.csv')
print(train.shape, test.shape)
train.head()

## Look at the data

In [ ]:
print(train.label.value_counts())
print()
print(train.groupby('label')[['amount', 'sim_age_hours', 'device_age_days', 'new_payee', 'pin_retries']].median().round(1))

In [ ]:
# Everything happens in time: payments per day, train vs test
pd.concat([train.timestamp.str[:10].value_counts().sort_index(), test.timestamp.str[:10].value_counts().sort_index()]).plot(figsize=(12, 3), title='payments per day');

## Single-row features
Only what is in the row itself, plus the sender's home city from `accounts.csv`. No information about other rows.

In [ ]:
home = accounts.set_index('account_id').home_city
CATS = ['receiver_type', 'txn_type', 'merchant_category', 'app']

def row_features(d):
    X = pd.DataFrame(index=d.index)
    t = pd.to_datetime(d.timestamp)
    X['hour'] = t.dt.hour
    X['weekday'] = t.dt.weekday
    for c in ['amount', 'sim_age_hours', 'device_age_days', 'sender_account_age_days', 'new_payee',
              'pin_retries', 'login_to_txn_sec', 'balance_before', 'qr_vpa_mismatch']:
        X[c] = d[c]
    X['amount_share_of_balance'] = d.amount / d.balance_before
    X['away_from_home'] = (d.device_city != d.sender_id.map(home)).astype(int)
    for c in CATS:
        X[c] = pd.Categorical(d[c].fillna('none'), categories=sorted(set(train[c].fillna('none')) | set(test[c].fillna('none')))).codes
    return X

X, Xt = row_features(train), row_features(test)
X.head()

## Validate, then train
A random split is the easy way, and it **overestimates** your score: test is a later month. A fairer check is to train on Jul–Aug and validate on September. Try both and compare.

In [ ]:
is_sep = train.timestamp.str[:7] == '2026-09'
model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=0)
model.fit(X[~is_sep], train.label[~is_sep])
pred = model.predict(X[is_sep])
print('macro-F1, train Jul-Aug -> validate Sep:', round(f1_score(train.label[is_sep], pred, average='macro'), 4))
print(classification_report(train.label[is_sep], pred, digits=3, zero_division=0))

In [ ]:
model.fit(X, train.label)
sub = pd.DataFrame({'id': test.txn_id, 'label': model.predict(Xt)})
sub.to_csv('submission.csv', index=False)
print(sub.label.value_counts())
print('wrote submission.csv', sub.shape)

## Ideas
Think like a fraud investigator. Each of these needs information from **other rows**, built with `groupby`, time windows or joins (use train **and** test together for the connections):

1. **Receivers:** how many different senders paid this receiver in the last hour or day? How fast does money leave again?
2. **Phones:** how many accounts used this `device_id`? How many payments in the last minute?
3. **Chains:** does the receiver send a slightly smaller amount to someone else within minutes?
4. **Pairs:** has this pair ever paid each other the other way? A ₹1 credit followed by a ₹5,000 payment back?
5. **Customers:** how many refund claims has this sender made this month?
6. **Amounts:** round numbers, and amounts just under a limit (₹1,999, ₹4,999, …).
7. **Merchants:** a sudden spike of round-number sales from the same few accounts.

And check every feature **still works in October** before you trust it. Good luck! 🕵️